# k sweep: MRE against the number of bins b = 5 … 14

The data behind `ksweep_mre_lines*.png` (and the dot and bar versions), as `paper_figures.ksweep` assembled it.
Each dataset's registry bin column (age wherever it has one) is binned into b bins by each method. The binned data
is then scored with each eval model on the seed's 30% test rows (`metrics.evaluate`), over 15 datasets × 9 seeds.

| Method | Cuts from | As recorded in |
|---|---|---|
| Fair Bin | each seed's training rows (stump tally, unbounded) | `robust_limits_stump_unbounded`, `rescore_ksweep` |
| Equal freq. (b bins) | each seed's training rows: sklearn quantile cuts, or `balanced_bins` where those give fewer than b bins | `robust_ksweep`, `rescore_ksweep` |
| Equal width, k-means, Jenks, Entropy | the whole column (`strategies.make_bins`) | `method_comparison`, `rescore_ksweep` |

Fair Bin and equal frequency put a value on a cut in the bin above (`[a, b)`); the other four, through `pd.cut`, in the
bin below (`(a, b]`).

**Run time:** many hours for all 15 datasets with all three models; the MLP on KDD dominates. Set `DATASETS` and
`MODELS` to a subset to try it.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import baselines, datasets, fairbin, figures, metrics

OUT = ROOT / "figures"
OUT.mkdir(exist_ok=True)
SEEDS = [92, 50, 61, 19, 11, 8, 2, 51, 70]  # the paper's nine eval seeds
DATASETS = datasets.group_names("uci")      # the 15 UCI datasets
KS = range(5, 15)
MODELS = ["rf", "xgb", "nn_early"]          # metrics.MODELS keys
INCREMENT = 0.001                           # outer-edge padding the whole-column baselines used

## Bins
Whole-column baselines are built once per (dataset, b), as `strategies.make_bins` built them; Fair Bin and equal frequency once per (dataset, seed, b) from the training rows.

In [ ]:
def whole_column_bins(method, data, spec, k):
    """(binned column, bounds) as strategies.make_bins made them: cuts from every row, realised with pd.cut."""
    col = data[spec.bin_col].copy()
    if method == "equal_width":
        binned, bounds, _ = baselines.gen_equal_bin(col, k, INCREMENT)
    elif method == "kmeans":
        binned, bounds, _ = baselines.gen_kmeans_bin(col, k, INCREMENT)
    elif method == "jenks":
        binned, bounds = baselines.gen_jenks_bin(col, k, INCREMENT)
    elif method == "entropy":
        binned, bounds = baselines.gen_entropy_bin_stop(col, data[spec.target], k, INCREMENT)
    return np.asarray(binned, dtype=float), [float(v) for v in bounds]


def eqfreq_k_cuts(train_col, k):
    """sklearn quantile cuts from the training rows; balanced_bins where they give fewer than k bins."""
    cuts = baselines.sk_quantile_bins(train_col, k)
    if len(cuts) - 1 < k:
        keys, cnt = np.unique(train_col.to_numpy(dtype=float), return_counts=True)
        cuts = baselines.balanced_bins(keys, cnt, k)
    return cuts


WHOLE = ["equal_width", "kmeans", "jenks", "entropy"]

## Run

In [ ]:
out = []
start = time.perf_counter()
for ds in DATASETS:
    data, spec, n_classes = datasets.load(ds)
    col = spec.bin_col
    whole = {(m, k): whole_column_bins(m, data, spec, k) for m in WHOLE for k in KS}
    for seed in SEEDS:
        train = data.loc[metrics.split(data, spec.target, seed)[0].index]
        values, errors, rows = fairbin.tally(train, spec.target, col, seed)
        z = fairbin.bonferroni_z(len(values))
        for k in KS:
            bins = dict({m: whole[(m, k)] for m in WHOLE})
            for m, cuts in (("fairbin_stump_nobound", fairbin.choose_cuts(values, errors, rows, k, z)[1]),
                            ("eqfreq_k", eqfreq_k_cuts(train[col], k))):
                bins[m] = (fairbin.apply_bins(data[col], cuts).to_numpy(float), cuts)
            for m, (binned, bounds) in bins.items():
                bin_idx, _ = metrics.bin_layout(binned, bounds)
                for model in MODELS:
                    ev = metrics.evaluate(data, col, spec.target, binned, bin_idx, len(bounds) - 1, model, seed,
                                          n_classes == 2)
                    out.append({"dataset": ds, "method": m, "k": k, "eval_model": model, "eval_seed": seed,
                                "mre": ev["mre"], "accuracy": ev["accuracy"], "bin_cuts": bounds})
        print(f"[{(time.perf_counter() - start) / 60:6.1f}m] {ds} seed {seed} done", flush=True)
runs = pd.DataFrame(out)

## Summary
MRE (%): per seed, the mean over datasets; then the mean over seeds.

In [ ]:
avg, order = figures.ksweep_avg(runs)
avg["mean"].unstack("k").round(1)

## Figures

In [ ]:
figures.pf_style()
figures.ksweep(runs, OUT)
for name in ("lines_rf", "lines_xgb", "lines_nn", "lines"):
    if (OUT / f"ksweep_mre_{name}.png").exists():
        display(Image(str(OUT / f"ksweep_mre_{name}.png")))